# Large Markdown Name

To make a name (or any text) look large in Markdown, use a single hash symbol `#` followed by a space at the start of a line. 

For example:
```markdown
# Your Name Here
```

In [28]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the State schema
class BMIState(TypedDict):
    weight_kg: float
    height: float
    bmi: float
    advice: str

# 2. Define the main processing nodes
def calculate_bmi(state: BMIState):
    weight = state['weight_kg']
    height = state['height']
    bmi = round(weight / (height ** 2), 2)
    return {"bmi": bmi}

def underweight_advice(state: BMIState):
    return {"advice": "We recommend a calorie-surplus diet and strength training."}

def normal_advice(state: BMIState):
    return {"advice": "Your BMI is in a healthy range. Maintain your current diet and lifestyle."}

def overweight_advice(state: BMIState):
    return {"advice": "We recommend a calorie-deficit diet and regular cardiovascular exercise."}

# 3. Define the Routing Function (Router/Routes function)
# This function inspects the state and returns the next route key/name
def route_based_on_bmi(state: BMIState) -> str:
    bmi = state.get("bmi", 0)
    if bmi < 18.5:
        return "underweight"
    elif bmi < 25.0:
        return "normal"
    else:
        return "overweight"

In [31]:
# 4. Construct the StateGraph
graph = StateGraph(BMIState)

# Add nodes to the graph
graph.add_node("calculate_bmi", calculate_bmi)
graph.add_node("underweight", underweight_advice)
graph.add_node("normal", normal_advice)
graph.add_node("overweight", overweight_advice)

# Add edge from START to the first node
graph.add_edge(START, "calculate_bmi")

# Add conditional edges using our routing function (route_based_on_bmi)
graph.add_conditional_edges(
    "calculate_bmi",
    route_based_on_bmi
)

# Route all terminal advice nodes to END
graph.add_edge("underweight", END)
graph.add_edge("normal", END)
graph.add_edge("overweight", END)

# Compile the workflow graph
workflow = graph.compile()

In [32]:
# 5. Run the Workflow with different test cases

# Test case 1: Healthy BMI
result_normal = workflow.invoke({"weight_kg": 70, "height": 1.75})
print("Healthy Case:", result_normal)

# Test case 2: Overweight BMI
result_overweight = workflow.invoke({"weight_kg": 90, "height": 1.75})
print("Overweight Case:", result_overweight)

Healthy Case: {'weight_kg': 70, 'height': 1.75, 'bmi': 22.86, 'advice': 'Your BMI is in a healthy range. Maintain your current diet and lifestyle.'}
Overweight Case: {'weight_kg': 90, 'height': 1.75, 'bmi': 29.39, 'advice': 'We recommend a calorie-deficit diet and regular cardiovascular exercise.'}
